In [1]:
!pip show langchain

Name: langchain
Version: 1.4.8
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [2]:
!pip install -qU langchain

In [4]:
!pip install -qU langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.3.0 which is incompatible.


In [5]:
from google.colab import userdata
from langchain.chat_models import init_chat_model

api_key = userdata.get("GEMINI_API_KEY")

model = init_chat_model(
    "gemini-3.5-flash-lite",
    model_provider="google_genai",
    api_key=api_key
)


In [6]:
!pip install -qU langchain-tavily

In [13]:
from langchain_tavily import TavilySearch
from pprint import pprint
tavily_api_key = userdata.get("TAVILY_API_KEY");

skill_demand_tool = TavilySearch(
    max_results=5,
    tavily_api_key=tavily_api_key,
    topic = "general",
    search_depth = "advanced"
)

In [19]:
import requests
from langchain.tools import tool

In [44]:
@tool
def search_jobs(skill:str, location:str) -> list:
  """Search for jobs based on skill and location"""

  print("Calling search jobs tool")
  print(f"Searching for {skill} jobs in {location}")

  url = "https://jsearch.p.rapidapi.com/search-v2"
  rapid_api_key = userdata.get("RAPID_API_KEY")

  querystring = {"query":f"{skill} in {location} for masters students","num_pages":"1","country":"us","date_posted":"all","employment_types":"INTERN"}

  headers = {
    "x-rapidapi-key": rapid_api_key,
    "x-rapidapi-host": "jsearch.p.rapidapi.com"
  }

  response = requests.get(url, headers=headers, params=querystring)
  data = response.json()

  # 1. Access the "jobs" list inside the "data" dictionary
  jobs = data.get("data", {}).get("jobs", [])
  print(f"Found {len(jobs)} with the {skill} & {location}")
  result = []

  for job in jobs:
      result.append(
          {
              # 2. Use dictionary bracket notation [] instead of dot notation .
              "title": job.get("job_title"),
              "company": job.get("employer_name"),
              "location": job.get("job_city"),
              "apply_link": job.get("job_apply_link")
          }
      )
  return result


In [45]:
from langchain.agents import create_agent

In [46]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [47]:
agent = create_agent(
    model = model,
    tools = [skill_demand_tool,search_jobs],
    system_prompt=system_prompt,
    debug = True
)

In [52]:
user_query = "What's the demand for software engineer in the industry and show me related job openings in India"


In [53]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_query
        }
    ]
})

[values] {'messages': [HumanMessage(content="What's the demand for software engineer in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='a63ba3a2-f0a2-4e28-8d5e-88ef145e7986')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "software engineer industry demand salary insights career trends India"}'}, '__gemini_function_call_thought_signatures__': {'call_4847487479722579054': 'EmAKXgFpFH0T74Hzi5SAEwWdqvbuFPQAuRQwqIXniwtn5gB1Mn5Q1y+8bbPky5hR++mvKfqFNTqFn+CorJnivZSeHy+7Yiot0ZopxJBmFy39xCiVf0PDS1ykohQ7fM+EyQM='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1d73b-09a0-75fd-ab39-c0f777f71401-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'software engineer industry demand salary insights career trends India'}, 'id': 'call

In [54]:
print(response["messages"][-1].content[0]['text'])

Industry Demand & Career Trends for Software Engineers in India

Demand Overview:
India's tech workforce is growing rapidly, employing over 5.8 million tech professionals. The overall tech sector is expanding at 10-15% annually. Hiring is particularly robust in Global Capability Centres (GCCs), product-first SaaS companies, fintech, and funded startups. 

Key In-Demand Specializations:
- AI, Machine Learning, and Generative AI (RAG pipelines, LLM fine-tuning) commanding 30% to 60% salary premiums over traditional roles.
- Cloud-Native & DevOps Engineering (Kubernetes, AWS, Terraform, Microservices).
- Full-Stack and Distributed Systems Architecture.

Salary Insights:
- Freshers: Typically start around 4 to 8 LPA (higher for top-tier product firms and IIT/NIT graduates).
- Mid-to-Senior Engineers: Range from 10 to 45+ LPA. Senior roles in tech hubs and GCCs can scale well past 50 LPA.
- Location Factor: Tech hubs like Bengaluru, Mumbai, and Hyderabad command significant pay premiums (15